# 🧩 04 · End-to-End 통합 평가 — 검색(01) × 생성(02 단일턴 · 03 멀티턴) 을 하나로
**GPT · GPU 를 쓰지 않습니다 → $0.** 01 · 02 · 03 이 `eval/results/` 에 저장한 평가 결과를 읽어서 합치기만 합니다.

```
01_rag_build      ── 🔎 검색 과정 평가 ─────────────▶ retrieval_01.json        (정답 문서 Hit@k · Recall · MRR · 키워드)
02_generate_eval  ── 🧠 생성 과정 평가 (단일턴) ─────▶ generation_02.json       (RAGAS + 숫자·문장 근거율 · 핵심 사실 활용 · 진단)
03_chatbot        ── 🧠 생성 과정 평가 (멀티턴) ─────▶ chatbot_03_<세션>.json   (직접 친 대화 · 대본 대화)
                                                          │
04_e2e_eval  ◀────────────────────────────────────────────┘  단계별 종합 · 질문별 통합표 · 실패 원인(검색 vs 생성) · 02 vs 03 비교
```
- **질문별로 이어 붙이기:** 02 · 03 의 질문이 01 정답 세트(`eval/retrieval_golden.jsonl`)에 있으면 01 의 검색 점수(Hit@k 등)가 붙습니다. 03 대본 대화의 후속 질문은 `gold` id 로 연결됩니다.
- **실패 원인 분류:** 질문마다 *검색이 근거를 가져왔나* × *답이 맞았나* × *근거에 충실했나* 로 `✅ 성공 · 🔎 검색 실패 · 🧠 생성 실패 · 📚 근거 없이 정답(일반지식) · 🙅 근거 있는데 거절 · ✅ 올바른 거절(문서 밖 질문)` 을 붙입니다.
- 순서: `0-1 → 1 → 2 → 3 → 4 → 5 → 6`. 01 · 02 · 03 중 일부만 있어도 있는 것만으로 평가합니다.
- 메모리를 거의 쓰지 않으므로 다른 노트북 커널을 끈 뒤 실행하세요 (VM RAM 16GB).

## 0. 경로 · 판정 기준
> 👥 **담당:** PM · 평가

In [1]:
# 0-1) 경로 · 판정 기준 — GPT · GPU 사용 없음 ($0)
import os, sys, time, importlib
from pathlib import Path

PROJECT_DIR = Path.cwd() if (Path.cwd() / "rag").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
import pandas as pd
from IPython.display import Markdown, display
import rag.stage_eval as SE
importlib.reload(SE)
pd.set_option("display.max_colwidth", 120)

RESULTS_DIR    = SE.RESULTS_DIR   # "eval/results"
INCLUDE_03     = "all"            # 합칠 03 세션: "all" | "latest"(가장 최근 1개) | "script"(대본 대화만) | "chat"(직접 친 대화만)
RETRIEVAL_MODE = "auto"           # 붙일 01 검색 결과: "auto"(02·03 설정과 같은 방식) | "hybrid" | "hybrid+rerank" | "dense" | "bm25" | "FAISS 직접"
K_JOIN         = 5                # 01 의 Hit@k · Recall@k 를 몇 위 기준으로 붙일지 (02 · 03 의 TOP_K 와 같게)
TH_RETRIEVAL   = 0.5              # 검색 성공: 핵심사실(근거) · Context Precision · 01 Hit@k 가 이 값 이상
TH_ANSWER      = 0.5              # 정답: Answer Correctness · 핵심사실(답변) 이 이 값 이상
TH_FAITH       = 0.5              # 근거 충실: Faithfulness · 숫자 근거율 · 문장 근거율 이 이 값 이상
print("결과 폴더:", (PROJECT_DIR / RESULTS_DIR).resolve(), "| 존재:", (PROJECT_DIR / RESULTS_DIR).exists())

결과 폴더: /home/spai1322/rfg_rag/eval/results | 존재: True


## 1. 결과 불러오기 — 01 · 02 · 03
> 👥 **담당:** 평가

In [2]:
# 1) 01 · 02 · 03 평가 결과 파일 읽기 + 목록
R01 = SE.load_result("retrieval_01", RESULTS_DIR)
G02 = SE.load_result("generation_02", RESULTS_DIR)
_p03 = SE.list_results("chatbot_03_", RESULTS_DIR)
if INCLUDE_03 == "script":
    _p03 = [p for p in _p03 if "_script_" in os.path.basename(p)]
elif INCLUDE_03 == "chat":
    _p03 = [p for p in _p03 if "_script_" not in os.path.basename(p)]
elif INCLUDE_03 == "latest":
    _p03 = _p03[-1:]
S03 = [(p, x) for p, x in ((p, SE.load_result(p)) for p in _p03) if x and x.get("rows")]


def _cfg_line(m):
    m = m or {}
    return (f"{m.get('llm_model')}·추론 {m.get('reasoning_effort')} · top_k {m.get('top_k')} · Dense {m.get('dense_weight')}"
            f" · 질문 해석 {'켜짐' if m.get('query_rewrite') else '꺼짐'} · 리랭커 {'켜짐' if m.get('rerank') else '꺼짐'}")


inv = []
if R01:
    m = R01.get("meta") or {}
    inv.append({"단계": "① 검색 (01)", "파일": "retrieval_01.json", "저장 시각": R01.get("saved_at"),
                "항목": f"질문 {len(R01.get('gold') or [])} × 방식 {len(R01.get('modes') or {})}",
                "주요 설정": f"{m.get('embed_model')} · chunk {m.get('chunk_size')} · Dense {(m.get('retrieval') or {}).get('dense_weight')} · 문서 {m.get('documents')}"})
if G02:
    inv.append({"단계": "② 생성 · 단일턴 (02)", "파일": "generation_02.json", "저장 시각": G02.get("saved_at"),
                "항목": f"질문 {len(G02['rows'])}", "주요 설정": _cfg_line(G02.get("meta")) + f" · 세트 {(G02.get('meta') or {}).get('question_sets')}"})
for p, x in S03:
    inv.append({"단계": "② 생성 · 멀티턴 (03)", "파일": os.path.basename(p), "저장 시각": x.get("saved_at"),
                "항목": f"{len(x['rows'])}턴 · 대화 {len({r.get('conv') for r in x['rows']})}개", "주요 설정": _cfg_line(x.get("meta"))})
display(pd.DataFrame(inv, columns=["단계", "파일", "저장 시각", "항목", "주요 설정"]).style.hide(axis="index").set_properties(**{"text-align": "left"}))
if not R01:
    print("ℹ️ 01 검색 평가 없음 → 01_rag_build 의 2-5 ~ 2-7 실행 (GPT 없음 · $0). 없으면 02 · 03 의 근거 지표로만 검색을 판단합니다")
if not G02:
    print("ℹ️ 02 생성 평가 없음 → 02_generate_eval 의 2-2 실행")
if not S03:
    print("ℹ️ 03 챗봇 결과 없음 → 03_chatbot 에서 SAVE_FOR_E2E=True 로 질문하거나, 2-4 대본 대화(RUN_SCRIPT=True) 실행")
_m02 = (G02 or {}).get("meta") or {}
_m03 = [(x.get("meta") or {}) for _, x in S03]
_keys = ["llm_model", "reasoning_effort", "top_k", "dense_weight", "query_rewrite", "rerank"]
_diff = [k for k in _keys if _m02 and any(m.get(k) != _m02.get(k) for m in _m03)]
if _diff:
    print(f"⚠️ 02 와 03 의 설정이 다릅니다 ({', '.join(_diff)}) → 02 vs 03 비교에서 차이가 설정 때문일 수 있습니다")

단계,파일,저장 시각,항목,주요 설정
① 검색 (01),retrieval_01.json,2026-10-12 07:12:23,질문 11 × 방식 4,nlpai-lab/KURE-v1 · chunk 700 · Dense 0.6 · 문서 98
② 생성 · 단일턴 (02),generation_02.json,2026-10-12 08:11:06,질문 20,gpt-5-mini·추론 low · top_k 5 · Dense 0.8 · 질문 해석 켜짐 · 리랭커 꺼짐 · 세트 ['함정 질문']
② 생성 · 멀티턴 (03),chatbot_03_20261012_081202.json,2026-10-12 08:12:52,1턴 · 대화 1개,gpt-5-mini·추론 low · top_k 5 · Dense 0.8 · 질문 해석 켜짐 · 리랭커 꺼짐


## 2. 🔎 ① 검색 단계 — 01 결과
> 질문 해석(GPT) 없이 *질문 그대로* 검색한 점수입니다. 02 · 03 은 질문 해석 + (리랭커)를 거치므로 실제 검색은 이보다 좋을 수 있습니다.

In [3]:
# 2) 01 검색 평가 요약 + E2E 에 붙일 검색 방식 고르기
R01_IDX, JOIN_MODE = {}, None
if R01:
    K = (R01.get("meta") or {}).get("k_main", K_JOIN)
    _cols = [f"Hit@{K}", f"Recall@{K}", "MRR", f"Precision@{K}", f"Keyword@{K}", "평균 검색(s)"]
    _sum = pd.DataFrame({m: v["summary"] for m, v in R01["modes"].items()}).T.reindex(columns=_cols).astype(float).rename_axis("검색 방식")
    _sc = [c for c in _cols if "(s)" not in c]
    display(Markdown(f"**검색 방식별 (k={K}, 문서 안 질문 평균)**"),
            _sum.style.format("{:.3f}", subset=_sc, na_rep="—").format("{:.2f}", subset=["평균 검색(s)"], na_rep="—")
            .background_gradient(subset=_sc, cmap="RdYlGn", vmin=0, vmax=1))
    if R01.get("sweep"):
        _sw = pd.DataFrame(R01["sweep"])
        _sc2 = [c for c in _sw.columns if "@" in c or c == "MRR"]
        display(Markdown("**Dense : BM25 가중치 스윕**"),
                _sw.style.format("{:.3f}", subset=_sc2, na_rep="—").background_gradient(subset=_sc2, cmap="RdYlGn", vmin=0, vmax=1).hide(axis="index"))
    if R01.get("separation"):
        print("문서 안 vs 문서 밖 질문 점수 분리:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in R01["separation"].items()})
    _want = RETRIEVAL_MODE
    if _want == "auto":
        _want = _m02.get("retrieval_mode") or (_m03[0].get("retrieval_mode") if _m03 else None) or "hybrid"
    JOIN_MODE = next(m for m in [_want, "hybrid", (R01.get("meta") or {}).get("main_mode"), *R01["modes"]] if m in R01["modes"])
    for r in R01["modes"][JOIN_MODE]["rows"]:
        R01_IDX[r["id"]] = r
        R01_IDX[SE.qkey(r["질문"])] = r
    print(f"→ E2E 에 붙일 01 검색 방식: {JOIN_MODE}" + ("" if JOIN_MODE == _want else f" ('{_want}' 결과가 없어 대신 사용)"))
else:
    print("01 결과 없음 — 건너뜀")

**검색 방식별 (k=5, 문서 안 질문 평균)**

,Hit@5,Recall@5,MRR,Precision@5,Keyword@5,평균 검색(s)
검색 방식,,,,,,
FAISS 직접,1.000,1.000,1.000,0.975,0.938,0.04
dense,1.000,1.000,1.000,0.925,1.000,0.43
bm25,1.000,1.000,1.000,0.925,1.000,0.40
hybrid,1.000,1.000,1.000,0.925,1.000,0.42


**Dense : BM25 가중치 스윕**

Dense 비율,BM25 비율,Hit@5,Recall@5,MRR,Precision@5,Keyword@5,평균 검색(s)
0.000000,1.000000,1.000,1.000,1.000,0.925,1.000,0.501727
0.200000,0.800000,1.000,1.000,1.000,0.925,1.000,0.391273
0.400000,0.600000,1.000,1.000,1.000,0.925,1.000,0.404909
0.600000,0.400000,1.000,1.000,1.000,0.925,1.000,0.396818
0.800000,0.200000,1.000,1.000,1.000,0.925,1.000,0.365909
1.000000,0.000000,1.000,1.000,1.000,0.925,1.000,0.381727


문서 안 vs 문서 밖 질문 점수 분리: {'문서 안 평균': 0.703, '문서 안 최소': 0.611, '문서 밖 평균': 0.503, '문서 밖 최대': 0.569, 'AUC': 1.0, '추천 임계값': 0.59, '판정': '✅ 완전히 분리됨'}
→ E2E 에 붙일 01 검색 방식: hybrid


## 3. 🧩 질문별 통합표 — 02 · 03 의 모든 질문에 01 검색 점수를 붙이고 실패 원인 분류
> 👥 **담당:** 평가

In [4]:
# 3) 질문별 통합 (02 단일턴 + 03 멀티턴) + 01 검색 점수 + E2E 판정
rows = []
if G02:
    rows += [{**r, "출처": "02 일괄(단일턴)", "세션": "02", "conv": r.get("id"), "turn": 1, "followup": False} for r in G02["rows"]]
for p, x in S03:
    _src = "03 대본(멀티턴)" if "_script_" in os.path.basename(p) else "03 챗봇(직접 질문)"
    rows += [{**r, "출처": _src, "세션": x.get("session") or os.path.basename(p)} for r in x["rows"]]
for r in rows:
    g = R01_IDX.get(r.get("gold_id")) or R01_IDX.get(SE.qkey(r.get("질문", "")))
    if g is not None:
        if g.get("type") == "ood":
            r["type"] = "ood"
        else:
            r["01 Hit@k"], r["01 Recall@k"] = g.get(f"Hit@{K_JOIN}"), g.get(f"Recall@{K_JOIN}")
            r["01 MRR"], r["01 Keyword@k"] = g.get("MRR"), g.get(f"Keyword@{K_JOIN}")
    r["type"] = r.get("type") or "in"
    r["검색 성공"] = SE.retrieval_ok(r, TH_RETRIEVAL)
    r["정답 여부"] = SE.answer_ok(r, TH_ANSWER)
    r["근거 충실"] = SE.grounded_ok(r, TH_FAITH)
    r["E2E 판정"] = SE.attribute(r, TH_RETRIEVAL, TH_ANSWER, TH_FAITH)
    r["E2E 성공"] = r["E2E 판정"].startswith("✅")
E2E = pd.DataFrame(rows)
if not len(E2E):
    raise SystemExit("합칠 결과가 없습니다 — 02 의 2-2 또는 03 을 먼저 실행하세요")
for c in ["거절", "문서 외 표시", "잘림", "followup", "E2E 성공"]:
    if c in E2E:
        E2E[c] = E2E[c].astype("boolean")
print(f"통합 {len(E2E)}건 = " + " · ".join(f"{k} {v}" for k, v in E2E["출처"].value_counts().items())
      + f" · 01 검색 점수가 붙은 질문 {int(E2E.get('01 Hit@k', pd.Series(dtype=float)).notna().sum())}건")

통합 21건 = 02 일괄(단일턴) 20 · 03 챗봇(직접 질문) 1 · 01 검색 점수가 붙은 질문 0건


## 4. 📊 단계별 종합 — 검색 · 생성 · E2E · 운영
> 출처(02 단일턴 · 03 멀티턴)별 평균. **↓ 표시 지표(거절 · 문서 외 · 잘림 · 근거 밖 사실률)는 낮을수록 좋습니다.**

In [5]:
# 4) 단계별 종합표 + 운영(응답 시간 · 비용)
LOWER_BETTER = {"거절", "문서 외 표시", "잘림", "근거 밖 사실률", "전체 응답(s)", "합계 $"}
STAGE = SE.stage_summary(E2E, by="출처")
STAGE.insert(2, "좋은 방향", STAGE["지표"].map(lambda m: "↓" if m in LOWER_BETTER else "↑"))
_groups = [c for c in STAGE.columns if c not in ("단계", "지표", "좋은 방향") and not c.endswith(" n")]
_quality = STAGE.index[STAGE["단계"] != "④ 운영"]
_up = STAGE.index[(STAGE["단계"] != "④ 운영") & (STAGE["좋은 방향"] == "↑")]
_down = STAGE.index[(STAGE["단계"] != "④ 운영") & (STAGE["좋은 방향"] == "↓")]
_sty = STAGE[["단계", "지표", "좋은 방향"] + _groups + [c for c in STAGE.columns if c.endswith(" n")]].style.format("{:.3f}", subset=_groups, na_rep="—")
if len(_up):
    _sty = _sty.background_gradient(subset=pd.IndexSlice[_up, _groups], cmap="RdYlGn", vmin=0, vmax=1)
if len(_down):
    _sty = _sty.background_gradient(subset=pd.IndexSlice[_down, _groups], cmap="RdYlGn_r", vmin=0, vmax=1)
display(_sty.hide(axis="index"))

def _p95(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return s.quantile(0.95) if len(s) else None
OPS = E2E.groupby("출처").agg(질문=("질문", "count"), 평균응답_s=("전체 응답(s)", lambda s: pd.to_numeric(s, errors="coerce").mean()),
                              p95응답_s=("전체 응답(s)", _p95), 질문당비용=("합계 $", lambda s: pd.to_numeric(s, errors="coerce").mean()),
                              E2E성공률=("E2E 성공", lambda s: s.astype(float).mean()))
display(Markdown("**운영 · 성공률** (비용 = 질문 해석 + 답변. RAGAS 심판 비용 제외)"),
        OPS.style.format({"평균응답_s": "{:.2f}", "p95응답_s": "{:.2f}", "질문당비용": "${:.5f}", "E2E성공률": "{:.1%}"}, na_rep="—")
        .background_gradient(subset=["E2E성공률"], cmap="RdYlGn", vmin=0, vmax=1))

단계,지표,좋은 방향,02 일괄(단일턴),03 챗봇(직접 질문),전체,02 일괄(단일턴) n,03 챗봇(직접 질문) n,전체 n
① 검색,핵심사실(근거),↑,0.650,—,0.650,20,0,20
① 검색,Context Precision,↑,0.590,0.000,0.562,20,1,21
② 생성,Faithfulness,↑,0.569,0.250,0.554,20,1,21
② 생성,Answer Relevancy,↑,0.555,0.977,0.575,20,1,21
② 생성,숫자 근거율,↑,0.929,—,0.929,14,0,14
② 생성,문장 근거율,↑,0.610,0.000,0.581,20,1,21
② 생성,근거 활용률,↑,1.000,—,1.000,13,0,13
② 생성,근거 밖 사실률,↓,0.071,—,0.071,14,0,14
② 생성,전제 교정,↑,1.000,—,1.000,5,0,5
② 생성,거절,↓,0.100,1.000,0.143,20,1,21


**운영 · 성공률** (비용 = 질문 해석 + 답변. RAGAS 심판 비용 제외)

,질문,평균응답_s,p95응답_s,질문당비용,E2E성공률
출처,,,,,
02 일괄(단일턴),20,3.78,7.16,$0.00219,30.0%
03 챗봇(직접 질문),1,27.69,27.69,$0.01074,0.0%


## 5. 🧭 실패 원인 — 검색 탓인가 생성 탓인가
> 👥 **담당:** 평가 → 결과를 검색 · 리랭킹 / 질문 해석 · 답변 담당에게 전달

In [6]:
# 5) 실패 원인 분류 (출처별 건수 · 비율) + 고칠 곳
ct = pd.crosstab(E2E["E2E 판정"], E2E["출처"], margins=True, margins_name="합계")
pct = ct.div(ct.loc["합계"], axis=1)
ATTR = ct.astype(str) + " (" + (pct * 100).round(0).astype(int).astype(str) + "%)"
display(ATTR.drop(index="합계").sort_values("합계", key=lambda s: -ct.drop(index="합계")["합계"]))

FIX = {
    "🔎 검색 실패": "검색 · 리랭킹 — 01 의 2-6 질문별 표에서 Hit 0 인 질문 확인 → DENSE_WEIGHT(2-7 추천값) · TOP_K · 리랭커 · 라우팅 · 정답 세트 패턴 점검",
    "🧠 생성 실패(근거는 있음)": "질문 해석 · 답변 — 근거는 왔는데 답이 틀림 → 프롬프트 · REASONING_EFFORT · ANSWER_WITH_REWRITE · MAX_TOKENS(잘림) 점검",
    "🧠 생성 실패(근거와 다른 답)": "질문 해석 · 답변 — 답이 근거와 다름(환각) → 프롬프트에 '근거에 없는 숫자 금지' · ALLOW_GENERAL 점검",
    "📚 근거 없이 정답(일반지식)": "데이터 — 질문의 답이 인덱스 문서에 없음(예: 연말정산 질문 ↔ RFP 인덱스). 문서를 추가하거나, 문서 밖 답변을 허용할지(ALLOW_GENERAL) 결정",
    "🙅 근거 있는데 거절": "답변 — 근거가 있는데 '확인할 수 없음' → 프롬프트의 거절 조건 완화 · 근거 문단 순서/길이 점검",
    "⚠️ 문서 밖 질문에 답변": "답변 — 문서 밖 질문에도 답함 → ALLOW_GENERAL=False 또는 관련도 게이트(01 의 점수 분리 · 추천 임계값)",
    "⚠️ 근거 충실하나 검색 미확인": "평가 — 정답 문서/핵심 사실이 없어 검색 성공을 확인 못함 → 01 정답 세트에 이 질문 추가",
    "❔ 판단 불가(정답·근거 지표 없음)": "평가 — 정답(REFERENCES) · 핵심 사실 · 정답 문서가 없음 → 02 의 0-4/0-5 또는 01 의 2-5 에 추가",
}
_fail = ct.drop(index="합계")["합계"].sort_values(ascending=False)
_fail = _fail[[not k.startswith("✅") for k in _fail.index]]
if len(_fail):
    display(Markdown("**🛠 고칠 곳 (많은 순)**\n\n" + "\n".join(f"{i}. **{k}** {v}건 — {FIX.get(k, '')}" for i, (k, v) in enumerate(_fail.items(), 1))))
else:
    print("🎉 실패로 분류된 질문이 없습니다")
if "유형" in E2E and E2E["유형"].notna().any():                 # 02 함정 질문 유형별 (오타 · 한문 · 한영 자판 · 영어 · RFP)
    _ty = E2E[E2E["유형"].notna()]
    display(Markdown("**함정 유형별 E2E 판정** — 어떤 유형에서 검색 실패 / 생성 실패가 몰리는지"),
            pd.crosstab(_ty["유형"], _ty["E2E 판정"], margins=True, margins_name="합계"))
    if "전제 교정" in _ty and _ty["전제 교정"].notna().any():
        _pp = _ty[_ty["전제 교정"].notna()]
        print(f"틀린 전제 질문 {len(_pp)}개 중 전제를 바로잡은 답변 {int(_pp['전제 교정'].astype(bool).sum())}개 "
              "(못 바로잡으면 E2E 판정에서 오답으로 처리)")

출처,02 일괄(단일턴),03 챗봇(직접 질문),합계
E2E 판정,,,
🧠 생성 실패(근거는 있음),7 (35%),0 (0%),7 (33%)
✅ 성공,6 (30%),0 (0%),6 (29%)
🔎 검색 실패,3 (15%),1 (100%),4 (19%)
📚 근거 없이 정답(일반지식),2 (10%),0 (0%),2 (10%)
🙅 근거 있는데 거절,2 (10%),0 (0%),2 (10%)


**🛠 고칠 곳 (많은 순)**

1. **🧠 생성 실패(근거는 있음)** 7건 — 질문 해석 · 답변 — 근거는 왔는데 답이 틀림 → 프롬프트 · REASONING_EFFORT · ANSWER_WITH_REWRITE · MAX_TOKENS(잘림) 점검
2. **🔎 검색 실패** 4건 — 검색 · 리랭킹 — 01 의 2-6 질문별 표에서 Hit 0 인 질문 확인 → DENSE_WEIGHT(2-7 추천값) · TOP_K · 리랭커 · 라우팅 · 정답 세트 패턴 점검
3. **📚 근거 없이 정답(일반지식)** 2건 — 데이터 — 질문의 답이 인덱스 문서에 없음(예: 연말정산 질문 ↔ RFP 인덱스). 문서를 추가하거나, 문서 밖 답변을 허용할지(ALLOW_GENERAL) 결정
4. **🙅 근거 있는데 거절** 2건 — 답변 — 근거가 있는데 '확인할 수 없음' → 프롬프트의 거절 조건 완화 · 근거 문단 순서/길이 점검

**함정 유형별 E2E 판정** — 어떤 유형에서 검색 실패 / 생성 실패가 몰리는지

E2E 판정,✅ 성공,📚 근거 없이 정답(일반지식),🔎 검색 실패,🙅 근거 있는데 거절,🧠 생성 실패(근거는 있음),합계
유형,,,,,,
영어 섞임,1,1,1,0,2,5
오타·띄어쓰기,4,0,0,0,1,5
한문 섞임,1,1,0,0,3,5
한영 자판,0,0,2,2,1,5
합계,6,2,3,2,7,20


틀린 전제 질문 5개 중 전제를 바로잡은 답변 5개 (못 바로잡으면 E2E 판정에서 오답으로 처리)


## 6. 🔁 같은 질문, 02(단일턴) vs 03(멀티턴)
> 같은 정답 세트 질문을 02 는 새 대화로, 03 대본은 대화를 이어서 물었습니다. 후속 질문(2턴~)에서 점수가 떨어지면 대화 맥락 처리(질문 해석 · HISTORY_TURNS)가 원인입니다.

In [7]:
# 6) 02 vs 03 비교 (gold id 또는 같은 질문 문장으로 연결) + 03 첫 질문 vs 후속 질문
E2E["비교키"] = [g if isinstance(g, str) and g else SE.qkey(q) for g, q in zip(E2E.get("gold_id", [None] * len(E2E)), E2E["질문"])]
E2E["단일/멀티"] = E2E["출처"].map(lambda s: "02 단일턴" if s.startswith("02") else "03 멀티턴")
_m = [c for c in ["Faithfulness", "Answer Correctness", "Context Precision", "숫자 근거율", "문장 근거율", "핵심사실(근거)",
                  "핵심사실(답변)", "근거 활용률", "01 Hit@k"] if c in E2E]
_both = E2E.groupby("비교키")["단일/멀티"].nunique()
_both = _both[_both > 1].index
if len(_both):
    _num = E2E[E2E["비교키"].isin(_both)].copy()
    for c in _m:
        _num[c] = pd.to_numeric(_num[c], errors="coerce")
    _num["E2E 성공"] = _num["E2E 성공"].astype(float)
    cmp = _num.groupby(["비교키", "단일/멀티"])[_m + ["E2E 성공"]].mean().unstack("단일/멀티")
    cmp.columns = [f"{a} · {b}" for a, b in cmp.columns]
    _q = _num.groupby("비교키").agg(질문=("질문", "first"), 후속질문=("followup", lambda s: bool(s.fillna(False).any())))
    cmp = _q.join(cmp)
    _vc = [c for c in cmp.columns if " · " in c]
    display(Markdown(f"**같은 질문 {len(cmp)}개 — 02 단일턴 vs 03 멀티턴**"),
            cmp.style.format("{:.2f}", subset=_vc, na_rep="—").background_gradient(subset=_vc, cmap="RdYlGn", vmin=0, vmax=1)
            .set_properties(subset=["질문"], **{"text-align": "left"}))
else:
    print("ℹ️ 02 와 03 에 같은 질문이 없습니다 → 02 의 QUESTION_SETS 에 'RFP' 를 넣고, 03 의 2-4 대본 대화(RUN_SCRIPT=True)를 실행하면 비교됩니다")
_m3 = E2E[E2E["출처"].str.startswith("03")]
if len(_m3) and _m3["followup"].fillna(False).any():
    _g = _m3.assign(구분=_m3["followup"].fillna(False).map({True: "후속 질문(2턴~)", False: "첫 질문"}))
    _mm = [c for c in _m if c in _g]
    for c in _mm:
        _g[c] = pd.to_numeric(_g[c], errors="coerce")
    _t = _g.groupby("구분")[_mm].mean()
    _t["E2E 성공률"] = _g.groupby("구분")["E2E 성공"].apply(lambda s: s.astype(float).mean())
    _t.insert(0, "턴 수", _g.groupby("구분").size())
    display(Markdown("**03: 첫 질문 vs 후속 질문**"),
            _t.style.format("{:.3f}", subset=_mm + ["E2E 성공률"], na_rep="—").background_gradient(subset=_mm + ["E2E 성공률"], cmap="RdYlGn", vmin=0, vmax=1))

ℹ️ 02 와 03 에 같은 질문이 없습니다 → 02 의 QUESTION_SETS 에 'RFP' 를 넣고, 03 의 2-4 대본 대화(RUN_SCRIPT=True)를 실행하면 비교됩니다


## 7. 📋 질문별 상세 + 저장
> 👥 **담당:** 평가 · PM

In [8]:
# 7) 질문별 상세 (실패 먼저) + 실패 답변 미리보기 + 저장 → eval/results/e2e_04*.csv · e2e_04.json
SHOW_FAILED_ANSWERS = True
_cols = [c for c in ["출처", "id", "질문", "followup", "E2E 판정", "진단", "검색 성공", "01 Hit@k", "정답 문서 근거", "핵심사실(근거)",
                     "Context Precision", "Faithfulness", "숫자 근거율", "문장 근거율", "근거 활용률", "Answer Correctness",
                     "핵심사실(답변)", "전체 응답(s)", "합계 $"] if c in E2E]
_num = [c for c in ["01 Hit@k", "핵심사실(근거)", "Context Precision", "Faithfulness", "숫자 근거율", "문장 근거율", "근거 활용률",
                    "Answer Correctness", "핵심사실(답변)"] if c in E2E]
DETAIL = E2E.sort_values(["E2E 성공", "출처", "id"], key=lambda s: s.astype(str))[_cols].copy()
for c in _num:
    DETAIL[c] = pd.to_numeric(DETAIL[c], errors="coerce")
display(DETAIL.style.format("{:.2f}", subset=_num, na_rep="—").format("{:.2f}", subset=[c for c in ["전체 응답(s)"] if c in DETAIL], na_rep="—")
        .format(lambda x: "—" if x != x or x is None else f"${x:.5f}", subset=[c for c in ["합계 $"] if c in DETAIL])
        .background_gradient(subset=_num, cmap="RdYlGn", vmin=0, vmax=1)
        .set_properties(subset=["질문", "E2E 판정"] + (["진단"] if "진단" in DETAIL else []), **{"text-align": "left"}).hide(axis="index"))
if SHOW_FAILED_ANSWERS:
    _f = E2E[~E2E["E2E 성공"].fillna(False)]
    if len(_f):
        display(Markdown("**실패 답변 미리보기**\n\n" + "\n\n".join(
            f"<details><summary><b>[{r['출처']}] {r.get('id')} · {r['E2E 판정']}</b> — {str(r['질문'])[:60]}</summary>\n\n"
            f"- 진단: {r.get('진단', '—')} · 근거 문서: {', '.join(str(s)[:30] for s in (r.get('sources') if isinstance(r.get('sources'), list) else [])[:3] if s) or '—'}\n"
            f"- 정답: {str(r.get('정답') or '—')[:200]}\n\n> {str(r.get('답변') or '')[:500].replace(chr(10), ' ')}\n\n</details>"
            for _, r in _f.iterrows())))

os.makedirs(RESULTS_DIR, exist_ok=True)
_drop = [c for c in ["contexts"] if c in E2E]
E2E.drop(columns=_drop).to_csv(os.path.join(RESULTS_DIR, "e2e_04_per_question.csv"), index=False, encoding="utf-8-sig")
STAGE.to_csv(os.path.join(RESULTS_DIR, "e2e_04_stage_summary.csv"), index=False, encoding="utf-8-sig")
_saved = SE.save_result("e2e_04", {
    "stage": "e2e", "notebook": "04_e2e_eval",
    "inputs": {"01": bool(R01), "02": bool(G02), "03": [os.path.basename(p) for p, _ in S03], "join_mode": JOIN_MODE},
    "thresholds": {"retrieval": TH_RETRIEVAL, "answer": TH_ANSWER, "faith": TH_FAITH, "k": K_JOIN},
    "stage_summary": STAGE, "ops": OPS.reset_index(), "attribution": ct.reset_index(),
    "rows": E2E.drop(columns=_drop)})
print("저장:", _saved, "·", os.path.join(RESULTS_DIR, "e2e_04_per_question.csv"), "·", os.path.join(RESULTS_DIR, "e2e_04_stage_summary.csv"))
print("E2E 성공률: " + " · ".join(f"{k} {v:.0%}" for k, v in E2E.groupby("출처")["E2E 성공"].apply(lambda s: s.astype(float).mean()).items())
      + "   💵 이 노트북의 GPT 사용: 없음 ($0)")

출처,id,질문,followup,E2E 판정,진단,검색 성공,정답 문서 근거,핵심사실(근거),Context Precision,Faithfulness,숫자 근거율,문장 근거율,근거 활용률,Answer Correctness,핵심사실(답변),전체 응답(s),합계 $
02 일괄(단일턴),Q04,수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?,False,🧠 생성 실패(근거는 있음),✅ 근거 반영,True,True,1.00,0.58,0.33,1.00,0.00,1.00,0.41,1.00,4.09,$0.00190
02 일괄(단일턴),Q07,서울시 地圖情報 플랫폼 高度化 事業費는 얼마?,False,🧠 생성 실패(근거는 있음),🔎 근거·답변 모두 핵심 사실 없음,True,True,0.00,0.00,0.57,—,0.00,—,0.12,0.00,2.44,$0.00115
02 일괄(단일턴),Q08,K-water 事業期間은 着手日로부터 90日이죠?,False,🧠 생성 실패(근거는 있음),🔎 근거·답변 모두 핵심 사실 없음,True,True,0.00,1.00,0.67,1.00,1.00,—,0.17,0.00,2.34,$0.00141
02 일괄(단일턴),Q09,을지대학교 瑕疵擔保 責任期間은?,False,🧠 생성 실패(근거는 있음),✅ 근거 반영,True,True,1.00,0.95,0.62,1.00,0.33,1.00,0.48,1.00,4.32,$0.00191
02 일괄(단일턴),Q10,우즈벡 灌漑시스템 用役期間은 언제까지?,False,📚 근거 없이 정답(일반지식),✅ 근거 반영,False,False,1.00,1.00,0.50,1.00,1.00,1.00,0.72,1.00,2.79,$0.00140
02 일괄(단일턴),Q11,dmfwleo tkdjqrlrks 언제까지야?,False,🔎 검색 실패,🔎 근거·답변 모두 핵심 사실 없음,False,False,0.00,0.00,0.03,1.00,0.53,—,0.15,0.00,8.70,$0.00392
02 일괄(단일턴),Q12,vkwn whdfidwpqhdxn 사업예산 알려줘,False,🙅 근거 있는데 거절,🙅 근거 있는데 거절,True,True,0.00,0.00,1.00,1.00,0.00,—,0.12,0.00,4.15,$0.00408
02 일괄(단일턴),Q13,tjdnftlflqeo 제안 발표시간 몇 분?,False,🧠 생성 실패(근거는 있음),✅ 근거 반영,True,True,1.00,0.87,1.00,1.00,0.83,1.00,0.41,1.00,4.97,$0.00532
02 일괄(단일턴),Q14,qpscjghkrdls 1ck dhvms 날짜가 언제야?,False,🔎 검색 실패,🔎 근거·답변 모두 핵심 사실 없음,False,False,0.00,0.00,0.33,1.00,0.50,—,0.10,0.00,3.04,$0.00285
02 일괄(단일턴),Q15,dlscjs ISP gkehrmq 돼요?,False,🙅 근거 있는데 거절,🙅 근거 있는데 거절,True,True,0.00,0.64,0.35,—,0.00,—,0.14,0.00,7.08,$0.00434


**실패 답변 미리보기**

<details><summary><b>[02 일괄(단일턴)] Q04 · 🧠 생성 실패(근거는 있음)</b> — 수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 , 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 , 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시
- 정답: 아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 하자보수보증금율은 계약금액의 2%이며, 하자담보 책임기간은 준공검사 완료일로부터 1년입니다.

> 정정: 수자원공사 사고분석솔루션 사업의 하자보수보증금율은 "계약금액의 2%"입니다. [2][3]

</details>

<details><summary><b>[02 일괄(단일턴)] Q07 · 🧠 생성 실패(근거는 있음)</b> — 서울시 地圖情報 플랫폼 高度化 事業費는 얼마?</summary>

- 진단: 🔎 근거·답변 모두 핵심 사실 없음 · 근거 문서: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계, 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계, 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계
- 정답: 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역의 사업비는 금 493,763천원(부가가치세 포함)입니다.

> 컨텍스트 문서에 사업비 금액이 명시되어 있지 않습니다. 문서의 관련 표들은 금액 칸이 비어 있거나(p.64) 계약금액이 자리표시자 형태로 되어 있습니다(p.64, p.62). [1][3][5]

</details>

<details><summary><b>[02 일괄(단일턴)] Q08 · 🧠 생성 실패(근거는 있음)</b> — K-water 事業期間은 着手日로부터 90日이죠?</summary>

- 진단: 🔎 근거·답변 모두 핵심 사실 없음 · 근거 문서: 한국수자원공사_건설통합시스템(CMS) 고도화.hwp 4, 한국수자원공사_건설통합시스템(CMS) 고도화.hwp 2, 한국수자원공사_건설통합시스템(CMS) 고도화.hwp 1
- 정답: 아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 사업기간은 착수일로부터 120일간입니다.

> 정정: K-water 사업기간은 착수일로부터 365일간 (12개월)입니다. [1]

</details>

<details><summary><b>[02 일괄(단일턴)] Q09 · 🧠 생성 실패(근거는 있음)</b> — 을지대학교 瑕疵擔保 責任期間은?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 을지대학교_을지대학교 비교과시스템 개발.hwp Ⅲ. 사, 을지대학교_을지대학교 비교과시스템 개발.hwp Ⅴ. 제, 을지대학교_을지대학교 비교과시스템 개발.hwp Ⅲ. 사
- 정답: 을지대학교 비교과시스템 개발 사업의 하자담보 책임기간은 사업을 종료한 날(최종산출물을 인도한 날)부터 1년입니다.

> 정정: 질문 전제(단일 기간)는 문서 내에서 상이합니다. 아래에 문서에 명시된 하자담보책임기간을 근거별로 정리합니다.  - "하자담보 책임기간은 을지대학교의 검수에 의하여 사업의 완성을 확인한 후 3년간으로 함" [요구사항 QUR-품질-006] [1].   - "무상 하자보수기간은 검사완료일로부터 36개월로 함" [요구사항 PSR-프로젝트지원-002] [3].   - 반면 제안서 작성요령·입찰조건 등 본문에서는 "하자담보책임기간은 사업을 종료한 날로부터 1년으로 함"이라고 명시되어 있음(복수 항목) [2][5].  따라서 문서들 간 상충이 있어 단일한 기간을 확정하려면 발주처(을지대학교)의 추가 확인이나 지침 우선순위 규정에 따른 해석이 필요합니다.

</details>

<details><summary><b>[02 일괄(단일턴)] Q10 · 📚 근거 없이 정답(일반지식)</b> — 우즈벡 灌漑시스템 用役期間은 언제까지?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화, KOICA 전자조달_[긴급] [지문] [국제] 우즈베키
- 정답: 우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업의 용역기간은 계약체결일로부터 2027년 12월 31일까지입니다.

> 우즈베키스탄 관개 시스템 용역기간은 "계약체결일로부터 2027년 12월 31일 까지"입니다. [1]

</details>

<details><summary><b>[02 일괄(단일턴)] Q11 · 🔎 검색 실패</b> — dmfwleo tkdjqrlrks 언제까지야?</summary>

- 진단: 🔎 근거·답변 모두 핵심 사실 없음 · 근거 문서: 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시스템 , 한국농수산식품유통공사_농산물가격안정기금 정부예산회계연계, 한국지식재산보호원_IP-NAVI  해외지식재산센터 사업
- 정답: 을지대학교 비교과시스템 개발 사업의 사업기간은 계약체결일로부터 2025년 4월 30일까지입니다.

> 요청하신 마감일(제출기한)을 문서별로 정리하면 다음과 같습니다.  - 서민금융진흥원(서민금융 채팅 상담시스템 구축): 전자입찰서 및 제안서 접수 마감은 '24.06.24. 입니다. 제안서 발표 및 기술평가는 접수 마감 후 7일 이내에 실시됩니다. [1]  - 한국지식재산보호원(IP-NAVI 해외지식재산센터): 사업 종료 후 1개월 이내에 최종 산출물(원본 소스코드, DB 백업자료 등)을 제출해야 하며, 일부 산출물은 사업 종료 14일 이전 제출로 명시되어 있습니다. [3]  - 한국생산기술연구원(2세대 전자조달시스템 기반구축사업):    - 착수계·보안관련 문서 등은 계약 후 7일 이내 제출입니다. [5]   - 사업수행계획서는 계약 후 10일 이내 제출입니다. [5]   - 착수보고서는 착수 후 10일 이내 제출입니다. [5]   - 완료보고서 및 최종산출물(USB 제출)은 계약종료 2주 전 까지 제출입니다. [5]   - 주간·월간보고는 매주, 매월 제출입니다. [5]  - 중앙

</details>

<details><summary><b>[02 일괄(단일턴)] Q12 · 🙅 근거 있는데 거절</b> — vkwn whdfidwpqhdxn 사업예산 알려줘</summary>

- 진단: 🙅 근거 있는데 거절 · 근거 문서: 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업, 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업, 파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업
- 정답: 파주도시관광공사 종량제봉투 판매관리 전산시스템 개선사업의 사업예산은 금 55,000,000원(부가세 포함)입니다.

> 문서에 사업예산(총액)이 명시되어 있지 않습니다. 제안 금액란이 빈칸으로 표시되어 있습니다: "제안 금액 일금 원 (￦ )" [4]. 또한 기간은 "착수일로부터 60일"로 명시되어 있습니다 [4]. 문서의 일부 표에는 숫자 '20'만 반복 표기되어 있으나 실제 금액으로 확인할 수 없습니다 [3][5].

</details>

<details><summary><b>[02 일괄(단일턴)] Q13 · 🧠 생성 실패(근거는 있음)</b> — tjdnftlflqeo 제안 발표시간 몇 분?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 재단법인 광주연구원_광주정책연구아카이브(GPA) 시스템, 서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 , 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 
- 정답: 서울시립대학교 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역의 제안 발표시간은 업체당 30분 내외(제안설명 20분, 질의응답 10분)입니다.

> 다음 문서들에서 제시한 제안 발표 시간은 각각 다음과 같습니다.  - (재단법인 광주연구원) 제안내용을 "20분 이내 설명하고 질의응답 10분 이내"로 함 [1].   - (서울시립대학교) "업체당 30분 내외(제안설명 20분, 질의응답 10분)"로 함 [2].   - (세종테크노파크) "제안사별 30분 이내(설명 15분, 질의응답 시간 15분)"로 함 [3].   - (고려대학교) "업체당 발표시간 : 50분 이내(발표 30분, 질의응답 20분, 사업 진행 상황에 따라 변경가능)"로 함 [4].   - (한국생산기술연구원) "시 간 : 제안사 당 30분 이내(15분 발표, 15분 질의응답)"로 함 [5].

</details>

<details><summary><b>[02 일괄(단일턴)] Q14 · 🔎 검색 실패</b> — qpscjghkrdls 1ck dhvms 날짜가 언제야?</summary>

- 진단: 🔎 근거·답변 모두 핵심 사실 없음 · 근거 문서: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) , 한국연구재단_2024년 기초학문자료센터 시스템 운영 및, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS)
- 정답: 벤처확인종합관리시스템 기능 고도화 사업의 1차 오픈은 2024년 7월 10일(성과조건부 및 스톡옵션 오픈)이며, 2차 오픈(Grand Open)은 2024년 9월 중입니다.

> 정정: 질문이 일부 오타가 있어 문의하신 문서 내 항목을 '착수보고서'로 해석하여 답합니다.  착수보고서는 "계약후 14일 이내", 국문으로 공문접수해야 합니다 [3].

</details>

<details><summary><b>[02 일괄(단일턴)] Q15 · 🙅 근거 있는데 거절</b> — dlscjs ISP gkehrmq 돼요?</summary>

- 진단: 🙅 근거 있는데 거절 · 근거 문서: 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립, 인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립
- 정답: 안 됩니다. 인천일자리플랫폼 정보시스템 구축 ISP 수립용역은 하도급을 허용하지 않습니다.

> 컨텍스트 문서에는 '인천 ISP 가입' 가능 여부(참여요건·신청절차·접수기간 등)에 대한 정보가 없습니다. 먼저 해당 사실을 알려드립니다. [문서에 해당 내용 없음]  다만 문서에 제안서 제출과 관련한 서약서·보안서약서 양식이 포함되어 있다는 내용은 확인됩니다: 제안서 제출 관련 서약서 및 보안서약서 양식이 제시되어 있습니다(별지 제16호·제17호 서식) 및 서약서에 주소·기관명·대표자 기재란이 있습니다 [2].  [문서 외 일반지식] - 일반적으로 지자체 RFP(제안요청서)에 참여하려면 공고문(또는 RFP)의 '응모자격', '제출서류', '제출기간·방법' 항목을 확인하고 해당 요건을 충족하여 제안서를 제출해야 합니다.   - 보안 관련 요구가 있는 용역의 경우 보안서약서 제출, 개인정보 처리·접근 통제 준수 등이 요구될 수 있으므로(제안서 양식에 보안서약서가 포함된 경우 특히) 관련 서류를 준비해야 합니다.   - 정확한 참여가능성(가입 여부), 자격요건, 제출기간 등은 해당 발주

</details>

<details><summary><b>[02 일괄(단일턴)] Q16 · 🧠 생성 실패(근거는 있음)</b> — K-water 사고분석솔루션 project budget이 VAT 별도 1억9,503만원 맞아?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 , 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 , 한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 
- 정답: 아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 사업예산 일금 195,030,000원은 부가가치세(VAT) 포함 금액입니다.

> 정정: 문서에는 사업금액이 "일금 일억구천오백삼만원정(￦195,030,000, VAT포함)"으로 명시되어 있어 VAT 포함 금액입니다 [4].

</details>

<details><summary><b>[02 일괄(단일턴)] Q17 · 📚 근거 없이 정답(일반지식)</b> — Uzbek irrigation 사업 consortium은 최대 몇 개사, minimum share는 몇 %?</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS), 부산관광공사_경영정보시스템 기능개선.hwp 3. 제안서
- 정답: 우즈벡-키르기즈스탄 스마트 관개시스템 구축사업의 공동수급체는 대표사 포함 5개사 이하이며, 참여 업체의 최소 지분율은 10% 이상입니다(공동이행·분담이행 모두 가능).

> Uzbek irrigation 사업 컨소시엄은 "공동도급 총 업체수는 대표사를 포함하여 5개사 이하"이며, 참여사별 "최소지분율은 10% 이상"입니다. [1]

</details>

<details><summary><b>[02 일괄(단일턴)] Q19 · 🔎 검색 실패</b> — Through which system should proposals be submitted for the 우</summary>

- 진단: ✅ 근거 반영 · 근거 문서: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화
- 정답: 제안서는 나라장터 e-발주시스템(http://rfp.g2b.go.kr)을 통해 온라인으로 제출하며, '제안서 제출 매뉴얼'을 참고합니다. (Proposals are submitted online through the 나라장터 e-발주시스템, http://rfp.g2b.go.kr.)

> 우즈벡-키르기즈스탄 관개 사업의 제안서는 나라장터의 e-발주시스템을 통한 온라인 제출입니다. [1]

</details>

<details><summary><b>[02 일괄(단일턴)] Q20 · 🧠 생성 실패(근거는 있음)</b> — 벤처확인 system 고도화 budget 3억5,200만원은 VAT excluded 맞죠?</summary>

- 진단: 📚 문서 외 일반지식으로 답변 · 근거 문서: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능, (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능, (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능
- 정답: 아닙니다. 벤처확인종합관리시스템 기능 고도화 용역사업의 소요예산 352,000,000원은 부가가치세(VAT) 포함 금액입니다.

> 컨텍스트 문서에는 예산 3억 5,200만원이 부가가치세 포함인지 여부에 대한 명시가 없습니다. 문서 내 관련 내용이 확인되지 않습니다.  [문서 외 일반지식] 관행적으로 공공 입찰·제안서에서는 금액이 '부가가치세 별도'인지 '부가가치세 포함'인지 명확히 표기합니다. 확인 방법은 다음과 같습니다. - 제안요청서 본문(예산·금액 표기), 계약서 또는 입찰공고의 금액 표시 문구를 확인하세요. (예: "금액은 부가가치세 별도" 또는 "부가가치세 포함") - 예산 관련 별지·붙임(예산서, 비용산정표 등)을 확인하세요. - 발주기관 담당자에게 서면으로 문의하여 확답을 받으세요.  원하시면 제가 문서 전체에서 '부가가치세', 'VAT', '포함', '별도' 같은 키워드를 찾아드리거나, 발주처에 질의할 문장 예시를 작성해 드리겠습니다.

</details>

<details><summary><b>[03 챗봇(직접 질문)] 20261012_081202-T1 · 🔎 검색 실패</b> — 너는 누구?</summary>

- 진단: 🙅 거절(근거 없음) · 근거 문서: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅., 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시, 전북특별자치도 정읍시_정읍체육트레이닝센터 통합운영관리시
- 정답: —

> 컨텍스트에 본 질문(“너는 누구?”)에 대한 정보가 없습니다.  [문서 외 일반지식] 저는 이 대화의 AI 어시스턴트입니다. 질문을 받고 답변을 제공하도록 설계되어 있으며, 필요하시면 어떻게 도와드릴지 말씀해 주세요.

</details>

저장: eval/results/e2e_04.json · eval/results/e2e_04_per_question.csv · eval/results/e2e_04_stage_summary.csv
E2E 성공률: 02 일괄(단일턴) 30% · 03 챗봇(직접 질문) 0%   💵 이 노트북의 GPT 사용: 없음 ($0)
